# Modelado: Predicción de Argumentos Efectivos

CC3084 Data Science, Universidad del Valle de Guatemala, Semestre II 2026

Reto 19, Feedback Prize - Predicting Effective Arguments.

Este notebook continúa el análisis exploratorio. Aquí se entrenan, se ajustan y se comparan los modelos.

## Decisiones que vienen del análisis exploratorio

**La partición se agrupa por ensayo.** Cada ensayo aporta 8.77 fragmentos en promedio que comparten autor, tema y nivel de escritura. Si se reparten al azar, el modelo vería fragmentos del mismo ensayo en entrenamiento y en prueba, y el resultado saldría inflado. Se usa GroupShuffleSplit para la partición y GroupKFold para la validación cruzada, siempre con `essay_id` como grupo.

**La métrica es log loss.** Adequate concentra el 57.06 % de los casos, así que la exactitud no distingue un modelo útil de uno que siempre predice la clase mayoritaria. Se reporta también F1 macro, que trata a las tres clases por igual.

**Se agregan variables del ensayo completo.** A nivel de fragmento la correlación más alta fue 0.289, pero a nivel de ensayo subió a 0.577. Por eso el modelo recibe también información del ensayo donde vive cada fragmento.

## Modelos que se comparan

**Línea base.** Predice siempre la distribución de las clases. Sirve de referencia mínima.

**TF-IDF con regresión logística.** Mide cuánta señal hay en las palabras.

**LightGBM sobre variables derivadas.** Mide cuánta señal hay en la estructura del ensayo.

**DeBERTa ajustado a la tarea.** Mide cuánta señal hay en el contexto. Va en la sección 10 porque necesita GPU.

## 0. Preparación del entorno

`sentencepiece` es necesario para el tokenizador de DeBERTa y no siempre viene instalado. Los números de versión sirven para ubicar rápido cualquier falla de compatibilidad.

In [ ]:
!pip install -q sentencepiece

import warnings
warnings.filterwarnings('ignore')

import transformers, sklearn, lightgbm
print('transformers', transformers.__version__)
print('sklearn', sklearn.__version__)
print('lightgbm', lightgbm.__version__)

In [ ]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import log_loss, f1_score, accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
AZUL, NARANJA, VERDE, ROJO = '#2F4B7C', '#E0A458', '#3E7C59', '#C1495B'

## 1. Carga y construcción de variables

In [ ]:
ruta = '/kaggle/input/competitions/feedback-prize-effectiveness'
if not os.path.exists(ruta):
    ruta = 'data/raw'

datos = pd.read_csv(f'{ruta}/train.csv')
datos['discourse_text'] = datos['discourse_text'].str.strip()

print(datos.shape)
datos.head()

In [ ]:
def construir_variables(tabla):
    tabla = tabla.copy()
    tabla['n_caracteres'] = tabla['discourse_text'].str.len()
    tabla['n_palabras'] = tabla['discourse_text'].str.split().str.len()
    tabla['n_oraciones'] = tabla['discourse_text'].str.count(r'[.!?]').clip(lower=1)
    tabla['n_comas'] = tabla['discourse_text'].str.count(',')
    tabla['largo_palabra'] = tabla['n_caracteres'] / tabla['n_palabras']
    tabla['palabras_por_oracion'] = tabla['n_palabras'] / tabla['n_oraciones']
    tabla['log_palabras'] = np.log1p(tabla['n_palabras'])

    tabla['posicion'] = tabla.groupby('essay_id').cumcount()
    tabla['total_fragmentos'] = tabla.groupby('essay_id')['discourse_id'].transform('count')
    tabla['posicion_relativa'] = (tabla['posicion'] / (tabla['total_fragmentos'] - 1)).fillna(0.5)

    tabla['palabras_ensayo'] = tabla.groupby('essay_id')['n_palabras'].transform('sum')
    tabla['tipos_ensayo'] = tabla.groupby('essay_id')['discourse_type'].transform('nunique')
    tabla['proporcion_del_ensayo'] = tabla['n_palabras'] / tabla['palabras_ensayo']
    tabla['palabras_vs_tipo'] = tabla['n_palabras'] / tabla.groupby('discourse_type')['n_palabras'].transform('median')

    tabla['tipo_anterior'] = tabla.groupby('essay_id')['discourse_type'].shift(1).fillna('INICIO')
    tabla['tipo_siguiente'] = tabla.groupby('essay_id')['discourse_type'].shift(-1).fillna('FIN')
    tabla['texto_completo'] = tabla['discourse_type'] + ' [SEP] ' + tabla['discourse_text']
    return tabla


datos = construir_variables(datos)

numericas = ['n_caracteres', 'n_palabras', 'n_oraciones', 'n_comas', 'largo_palabra',
             'palabras_por_oracion', 'log_palabras', 'posicion', 'total_fragmentos',
             'posicion_relativa', 'palabras_ensayo', 'tipos_ensayo',
             'proporcion_del_ensayo', 'palabras_vs_tipo']

categoricas = ['discourse_type', 'tipo_anterior', 'tipo_siguiente']

datos[numericas].describe().T.round(2)

Cuatro variables son nuevas respecto al análisis exploratorio y las cuatro salen de hallazgos concretos.

**`palabras_vs_tipo`** compara la longitud del fragmento contra la mediana de su propio tipo. Sale del hallazgo de que Evidence promedia 77.73 palabras y Claim solo 18.74, así que la longitud absoluta no se puede comparar entre tipos.

**`proporcion_del_ensayo`** mide cuánto del ensayo ocupa el fragmento. Distingue al estudiante que escribe poco en todo del que escribió un fragmento corto dentro de un ensayo largo.

**`tipo_anterior` y `tipo_siguiente`** dan el contexto de los fragmentos vecinos. Salen del hallazgo de que el mismo texto puede tener etiquetas distintas según el ensayo donde aparece.

## 2. Partición agrupada por ensayo

In [ ]:
clases = ['Ineffective', 'Adequate', 'Effective']
objetivo = datos['discourse_effectiveness'].map({c: i for i, c in enumerate(clases)}).values
grupos = datos['essay_id'].values

separador = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indice_entrena, indice_prueba = next(separador.split(datos, objetivo, grupos))

entrena = datos.iloc[indice_entrena].reset_index(drop=True)
prueba = datos.iloc[indice_prueba].reset_index(drop=True)
y_entrena = objetivo[indice_entrena]
y_prueba = objetivo[indice_prueba]

interno = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
indice_ajuste, indice_validacion = next(interno.split(entrena, y_entrena, entrena['essay_id']))

ajuste = entrena.iloc[indice_ajuste].reset_index(drop=True)
validacion = entrena.iloc[indice_validacion].reset_index(drop=True)
y_ajuste = y_entrena[indice_ajuste]
y_validacion = y_entrena[indice_validacion]

print('entrenamiento:', entrena.shape, 'ensayos:', entrena['essay_id'].nunique())
print('prueba:', prueba.shape, 'ensayos:', prueba['essay_id'].nunique())
print('ensayos compartidos entre entrenamiento y prueba:',
      len(set(entrena['essay_id']) & set(prueba['essay_id'])))
print()
print('ajuste:', ajuste.shape, 'ensayos:', ajuste['essay_id'].nunique())
print('validación:', validacion.shape, 'ensayos:', validacion['essay_id'].nunique())
print('ensayos compartidos entre ajuste y validación:',
      len(set(ajuste['essay_id']) & set(validacion['essay_id'])))

Los dos conteos de ensayos compartidos deben dar cero. Esa es la comprobación de que ninguna partición mezcla fragmentos del mismo ensayo.

Hay tres conjuntos y cada uno tiene una función distinta. **Ajuste** entrena los modelos mientras se buscan los parámetros. **Validación** mide cada combinación de parámetros y decide cuál gana. **Prueba** no se toca hasta el final y solo sirve para reportar el resultado. Si se usara el conjunto de prueba para elegir parámetros, el número final saldría mejor de lo que realmente es.

## 3. Línea base

In [ ]:
resultados = {}


def evaluar(nombre, probabilidades, segundos):
    predicciones = probabilidades.argmax(axis=1)
    resultados[nombre] = {
        'log_loss': float(log_loss(y_prueba, probabilidades, labels=[0, 1, 2])),
        'f1_macro': float(f1_score(y_prueba, predicciones, average='macro')),
        'exactitud': float(accuracy_score(y_prueba, predicciones)),
        'segundos': float(segundos)
    }
    print(nombre)
    for clave, valor in resultados[nombre].items():
        print(f'  {clave}: {valor:.4f}')
    return predicciones


vacio_entrena = np.zeros((len(entrena), 1))
vacio_prueba = np.zeros((len(prueba), 1))

inicio = time.time()
base = DummyClassifier(strategy='prior').fit(vacio_entrena, y_entrena)
evaluar('Línea base', base.predict_proba(vacio_prueba), time.time() - inicio)

## 4. Ajuste de parámetros

Toda esta sección trabaja sobre ajuste y validación. El conjunto de prueba no se usa aquí.

### 4.1 Regresión logística, búsqueda del parámetro de regularización

In [ ]:
vectorizador_ajuste = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=60000,
                                      sublinear_tf=True, strip_accents='unicode')
texto_ajuste = vectorizador_ajuste.fit_transform(ajuste['texto_completo'])
texto_validacion = vectorizador_ajuste.transform(validacion['texto_completo'])

busqueda_regresion = []

for valor_c in [0.25, 0.5, 1.0, 2.0, 4.0, 8.0]:
    inicio = time.time()
    modelo = LogisticRegression(C=valor_c, max_iter=1500)
    modelo.fit(texto_ajuste, y_ajuste)
    perdida = log_loss(y_validacion, modelo.predict_proba(texto_validacion), labels=[0, 1, 2])
    busqueda_regresion.append({'C': valor_c, 'log_loss': perdida, 'segundos': time.time() - inicio})
    print(f'C={valor_c}: log loss {perdida:.4f}')

busqueda_regresion = pd.DataFrame(busqueda_regresion).sort_values('log_loss')
mejor_c = float(busqueda_regresion.iloc[0]['C'])
print()
print('mejor C:', mejor_c)
busqueda_regresion.round(4)

### 4.2 LightGBM, búsqueda aleatoria

In [ ]:
reductor_ajuste = TruncatedSVD(n_components=150, random_state=42)
codificador_ajuste = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
escalador_ajuste = StandardScaler()

tabla_ajuste = np.hstack([
    escalador_ajuste.fit_transform(ajuste[numericas]),
    codificador_ajuste.fit_transform(ajuste[categoricas]),
    reductor_ajuste.fit_transform(texto_ajuste)
])

tabla_validacion = np.hstack([
    escalador_ajuste.transform(validacion[numericas]),
    codificador_ajuste.transform(validacion[categoricas]),
    reductor_ajuste.transform(texto_validacion)
])

print('columnas del modelo:', tabla_ajuste.shape[1])

In [ ]:
import lightgbm as lgb

espacio = {
    'num_leaves': [31, 63, 95, 127],
    'learning_rate': [0.02, 0.05, 0.08],
    'min_child_samples': [20, 40, 80],
    'colsample_bytree': [0.5, 0.7, 0.9],
    'reg_lambda': [0.0, 1.0, 5.0]
}

azar = np.random.RandomState(42)
busqueda_bosque = []

for intento in range(10):
    parametros = {clave: azar.choice(valores).item() for clave, valores in espacio.items()}
    inicio = time.time()

    candidato = lgb.LGBMClassifier(
        objective='multiclass', num_class=3, n_estimators=800,
        subsample=0.8, subsample_freq=1, random_state=42, verbose=-1, **parametros)

    candidato.fit(tabla_ajuste, y_ajuste,
                  eval_set=[(tabla_validacion, y_validacion)],
                  eval_metric='multi_logloss',
                  callbacks=[lgb.early_stopping(50, verbose=False)])

    perdida = log_loss(y_validacion, candidato.predict_proba(tabla_validacion), labels=[0, 1, 2])
    registro = dict(parametros)
    registro.update({'arboles': candidato.best_iteration_, 'log_loss': perdida,
                     'segundos': time.time() - inicio})
    busqueda_bosque.append(registro)
    print(f'intento {intento + 1:2d}: log loss {perdida:.4f}  {parametros}')

busqueda_bosque = pd.DataFrame(busqueda_bosque).sort_values('log_loss').reset_index(drop=True)
enteros = ['num_leaves', 'min_child_samples']
mejores_parametros = busqueda_bosque.iloc[0].drop(['arboles', 'log_loss', 'segundos']).to_dict()
mejores_parametros = {k: int(v) if k in enteros else float(v) for k, v in mejores_parametros.items()}
mejores_arboles = int(busqueda_bosque.iloc[0]['arboles'])

print()
print('mejores parámetros:', mejores_parametros)
print('árboles:', mejores_arboles)
busqueda_bosque.round(4)

### 4.3 Resumen del ajuste

In [ ]:
figura, ejes = plt.subplots(1, 2, figsize=(15, 4.5))

ejes[0].plot(busqueda_regresion.sort_values('C')['C'],
             busqueda_regresion.sort_values('C')['log_loss'],
             marker='o', color=AZUL)
ejes[0].set_xscale('log')
ejes[0].set_xlabel('C, inverso de la regularización')
ejes[0].set_ylabel('log loss en validación')
ejes[0].set_title('Regresión logística, búsqueda de C')

orden = busqueda_bosque.sort_values('log_loss', ascending=False)
etiquetas = [f"hojas {int(f.num_leaves)}, lr {f.learning_rate}" for f in orden.itertuples()]
ejes[1].barh(etiquetas, orden['log_loss'], color=AZUL)
ejes[1].set_xlabel('log loss en validación')
ejes[1].set_title('LightGBM, búsqueda aleatoria de 10 combinaciones')
ejes[1].set_xlim(orden['log_loss'].min() * 0.98, orden['log_loss'].max() * 1.01)

plt.tight_layout()
plt.show()

## 5. Modelos finales

Con los parámetros elegidos, los modelos se vuelven a entrenar sobre todo el conjunto de entrenamiento, que incluye ajuste y validación juntos, y se evalúan contra el conjunto de prueba.

### 5.1 TF-IDF con regresión logística

In [ ]:
inicio = time.time()

vectorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=60000,
                               sublinear_tf=True, strip_accents='unicode')
X_entrena = vectorizador.fit_transform(entrena['texto_completo'])
X_prueba = vectorizador.transform(prueba['texto_completo'])

regresion = LogisticRegression(C=mejor_c, max_iter=1500)
regresion.fit(X_entrena, y_entrena)

evaluar('TF-IDF + Regresión logística', regresion.predict_proba(X_prueba), time.time() - inicio)
print('vocabulario:', len(vectorizador.vocabulary_))

### 5.2 LightGBM

In [ ]:
inicio = time.time()

reductor = TruncatedSVD(n_components=150, random_state=42)
codificador = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
escalador = StandardScaler()

tabla_entrena = np.hstack([
    escalador.fit_transform(entrena[numericas]),
    codificador.fit_transform(entrena[categoricas]),
    reductor.fit_transform(X_entrena)
])

tabla_prueba = np.hstack([
    escalador.transform(prueba[numericas]),
    codificador.transform(prueba[categoricas]),
    reductor.transform(X_prueba)
])

bosque = lgb.LGBMClassifier(
    objective='multiclass', num_class=3, n_estimators=mejores_arboles,
    subsample=0.8, subsample_freq=1, random_state=42, verbose=-1, **mejores_parametros)

bosque.fit(tabla_entrena, y_entrena)

evaluar('LightGBM', bosque.predict_proba(tabla_prueba), time.time() - inicio)

## 6. Validación cruzada agrupada

El resultado sobre una sola partición puede ser suerte. Con GroupKFold de 5 pliegues se ve qué tan estable es el modelo.

In [ ]:
def validar(nombre, construir, n_pliegues=5):
    pliegues = GroupKFold(n_splits=n_pliegues)
    perdidas = []

    for numero, (idx_a, idx_b) in enumerate(pliegues.split(datos, objetivo, grupos), start=1):
        probabilidades = construir(datos.iloc[idx_a], objetivo[idx_a], datos.iloc[idx_b])
        perdida = log_loss(objetivo[idx_b], probabilidades, labels=[0, 1, 2])
        perdidas.append(perdida)
        print(f'  pliegue {numero}: {perdida:.4f}')

    print(f'{nombre}: {np.mean(perdidas):.4f} +/- {np.std(perdidas):.4f}')
    return perdidas


def construir_tfidf(a, y_a, b):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=60000, sublinear_tf=True)
    modelo = LogisticRegression(C=mejor_c, max_iter=1500)
    modelo.fit(vec.fit_transform(a['texto_completo']), y_a)
    return modelo.predict_proba(vec.transform(b['texto_completo']))


perdidas_tfidf = validar('TF-IDF + Regresión logística', construir_tfidf)

## 7. Comparación de modelos

In [ ]:
comparacion = pd.DataFrame(resultados).T.sort_values('log_loss')
comparacion.round(4)

In [ ]:
figura, ejes = plt.subplots(1, 3, figsize=(16, 4.5))

for eje, metrica, titulo, menor_mejor, color in zip(
    ejes,
    ['log_loss', 'f1_macro', 'segundos'],
    ['Log loss, menor es mejor', 'F1 macro, mayor es mejor', 'Tiempo de entrenamiento en segundos'],
    [True, False, True],
    [AZUL, VERDE, NARANJA]
):
    orden = comparacion.sort_values(metrica, ascending=not menor_mejor)
    eje.barh(orden.index, orden[metrica], color=color)
    eje.set_title(titulo)
    for posicion, valor in enumerate(orden[metrica]):
        eje.text(valor, posicion, f' {valor:.3f}', va='center', fontsize=9)
    eje.set_xlim(0, orden[metrica].max() * 1.25)

plt.tight_layout()
plt.show()

In [ ]:
mejor = comparacion.index[0]
probabilidades = {
    'TF-IDF + Regresión logística': lambda: regresion.predict_proba(X_prueba),
    'LightGBM': lambda: bosque.predict_proba(tabla_prueba)
}.get(mejor, lambda: base.predict_proba(vacio_prueba))()

predicciones = probabilidades.argmax(axis=1)
matriz = confusion_matrix(y_prueba, predicciones, normalize='true') * 100

plt.figure(figsize=(6.5, 5))
sns.heatmap(matriz.round(1), annot=True, fmt='.1f', cmap='Blues',
            xticklabels=clases, yticklabels=clases, cbar=False)
plt.title(f'Matriz de confusión, {mejor} (% por fila)')
plt.ylabel('clase real')
plt.xlabel('clase predicha')
plt.tight_layout()
plt.show()

print(classification_report(y_prueba, predicciones, target_names=clases, digits=3, zero_division=0))

In [ ]:
nombres_columnas = (numericas
                    + list(codificador.get_feature_names_out(categoricas))
                    + [f'texto_{i}' for i in range(150)])

importancias = pd.Series(bosque.feature_importances_, index=nombres_columnas)
importancias = importancias.sort_values(ascending=False).head(20)

plt.figure(figsize=(9, 6))
plt.barh(importancias.index[::-1], importancias.values[::-1], color=AZUL)
plt.title('LightGBM, 20 variables más usadas')
plt.xlabel('veces que el modelo usó la variable para partir')
plt.tight_layout()
plt.show()

El gráfico de importancias sirve para conectar el modelo con el análisis exploratorio. Si las variables de ensayo aparecen arriba, confirma el hallazgo de que la calidad es una propiedad del ensayo completo.

## 8. Guardado para la aplicación

In [ ]:
salida = '/kaggle/working' if os.path.exists('/kaggle/working') else 'modelos'
os.makedirs(salida, exist_ok=True)

joblib.dump(vectorizador, f'{salida}/vectorizador.joblib')
joblib.dump(regresion, f'{salida}/regresion.joblib')
joblib.dump({'reductor': reductor, 'codificador': codificador,
             'escalador': escalador, 'modelo': bosque}, f'{salida}/lightgbm.joblib')

with open(f'{salida}/metricas.json', 'w') as archivo:
    json.dump(resultados, archivo, indent=2)

referencias = {
    'mediana_por_tipo': datos.groupby('discourse_type')['n_palabras'].median().to_dict(),
    'mediana_ensayo': {columna: float(datos[columna].median()) for columna in
                       ['palabras_ensayo', 'total_fragmentos', 'tipos_ensayo',
                        'proporcion_del_ensayo', 'posicion', 'posicion_relativa']}
}

with open(f'{salida}/referencias.json', 'w') as archivo:
    json.dump(referencias, archivo, indent=2)

busqueda_regresion.to_csv(f'{salida}/busqueda_regresion.csv', index=False)
busqueda_bosque.to_csv(f'{salida}/busqueda_lightgbm.csv', index=False)
comparacion.to_csv(f'{salida}/comparacion_modelos.csv')

print(sorted(os.listdir(salida)))

## 9. Resumen de los parámetros ajustados

Esta tabla es la que va en la sección de resultados del informe, donde se pide explicar el ajuste de parámetros de cada algoritmo.

In [ ]:
resumen_ajuste = pd.DataFrame([
    {'Modelo': 'TF-IDF + Regresión logística', 'Parámetro': 'C',
     'Valores probados': '0.25, 0.5, 1, 2, 4, 8', 'Elegido': mejor_c},
    {'Modelo': 'LightGBM', 'Parámetro': 'num_leaves',
     'Valores probados': '31, 63, 95, 127', 'Elegido': mejores_parametros['num_leaves']},
    {'Modelo': 'LightGBM', 'Parámetro': 'learning_rate',
     'Valores probados': '0.02, 0.05, 0.08', 'Elegido': mejores_parametros['learning_rate']},
    {'Modelo': 'LightGBM', 'Parámetro': 'min_child_samples',
     'Valores probados': '20, 40, 80', 'Elegido': mejores_parametros['min_child_samples']},
    {'Modelo': 'LightGBM', 'Parámetro': 'colsample_bytree',
     'Valores probados': '0.5, 0.7, 0.9', 'Elegido': mejores_parametros['colsample_bytree']},
    {'Modelo': 'LightGBM', 'Parámetro': 'reg_lambda',
     'Valores probados': '0, 1, 5', 'Elegido': mejores_parametros['reg_lambda']},
    {'Modelo': 'LightGBM', 'Parámetro': 'n_estimators',
     'Valores probados': 'hasta 800 con parada temprana', 'Elegido': mejores_arboles},
])

resumen_ajuste

## 10. Modelo 3, DeBERTa

Este bloque necesita GPU. En Kaggle se enciende en el panel derecho, bajo Session options, Accelerator. Tarda cerca de una hora.

La idea es la misma que dominó la competencia. Al modelo se le pasa el tipo de argumento junto con el texto del fragmento, separados por un token especial, para que aprenda la relación entre los dos.

Para correrlo hay que cambiar la bandera a `True`.

La llamada a `.float()` sobre el modelo evita el error de gradientes en media precisión. Sin ella, los pesos llegan en fp16 y el escalador de gradientes falla. Si aun así falla, se pone `fp16=False`, que entrena más lento pero siempre funciona.

In [ ]:
ENTRENAR_TRANSFORMER = False

if ENTRENAR_TRANSFORMER:
    import torch
    from torch.utils.data import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)

    modelo_base = 'microsoft/deberta-v3-base'
    tokenizador = AutoTokenizer.from_pretrained(modelo_base)

    class Fragmentos(Dataset):
        def __init__(self, tabla, etiquetas):
            self.textos = (tabla['discourse_type'] + ' ' + tokenizador.sep_token + ' ' +
                           tabla['discourse_text']).tolist()
            self.etiquetas = etiquetas

        def __len__(self):
            return len(self.textos)

        def __getitem__(self, indice):
            codificado = tokenizador(self.textos[indice], truncation=True,
                                     max_length=256, padding='max_length')
            codificado = {k: torch.tensor(v) for k, v in codificado.items()}
            codificado['labels'] = torch.tensor(self.etiquetas[indice])
            return codificado

    def metricas(prediccion):
        logits, etiquetas = prediccion
        probabilidades = torch.softmax(torch.tensor(logits), dim=1).numpy()
        return {
            'log_loss': log_loss(etiquetas, probabilidades, labels=[0, 1, 2]),
            'f1_macro': f1_score(etiquetas, probabilidades.argmax(axis=1), average='macro')
        }

    transformer = AutoModelForSequenceClassification.from_pretrained(modelo_base, num_labels=3)
    transformer = transformer.float()

    configuracion = TrainingArguments(
        output_dir='deberta', num_train_epochs=2,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=2e-5, warmup_ratio=0.1, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='log_loss',
        greater_is_better=False, fp16=True, report_to='none')

    entrenador = Trainer(
        model=transformer, args=configuracion,
        train_dataset=Fragmentos(ajuste, y_ajuste),
        eval_dataset=Fragmentos(validacion, y_validacion),
        compute_metrics=metricas)

    inicio = time.time()
    entrenador.train()
    salida_prueba = entrenador.predict(Fragmentos(prueba, y_prueba))
    probabilidades_deberta = torch.softmax(torch.tensor(salida_prueba.predictions), dim=1).numpy()
    evaluar('DeBERTa', probabilidades_deberta, time.time() - inicio)

    comparacion = pd.DataFrame(resultados).T.sort_values('log_loss')
    display(comparacion.round(4))

    with open(f'{salida}/metricas.json', 'w') as archivo:
        json.dump(resultados, archivo, indent=2)
    comparacion.to_csv(f'{salida}/comparacion_modelos.csv')
    print('métricas actualizadas con los tres modelos')

## 11. Pendientes

**Correr el bloque de DeBERTa** con GPU encendida y volver a exportar `metricas.json`, para que la aplicación muestre los tres modelos.

**Medir el tiempo de inferencia** además del de entrenamiento, si se quiere ampliar la comparación entre algoritmos.

**Conectar los artefactos guardados** con la aplicación de Streamlit que está en la carpeta `app`.